In [1]:
import bcrypt

# Simple in-memory "database"
database = {}

# Registration function
def register(username, password):
    if username in database:
        print("User already exists!")
        return
    
    password_bytes = password.encode('utf-8')
    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(password_bytes, salt)
    
    database[username] = hashed
    print("User registered successfully!")

# Login function
def login(username, password):
    if username not in database:
        print("User not found!")
        return
    
    stored_hash = database[username]
    password_bytes = password.encode('utf-8')
    
    if bcrypt.checkpw(password_bytes, stored_hash):
        print("Login successful!")
    else:
        print("Login failed! Wrong password.")

register("alice", "rakshit")
login("alice", "rakshit")  # correct
login("alice", "umuthan")   # incorrect


User registered successfully!
Login successful!
Login failed! Wrong password.


In [2]:
import bcrypt
database = {}

def register(username, password):
    password_bytes = password.encode('utf-8')  # UTF-8 handles emojis
    hashed = bcrypt.hashpw(password_bytes, bcrypt.gensalt())
    database[username] = hashed
    print("Registered successfully!")

def login(username, password):
    password_bytes = password.encode('utf-8')
    if bcrypt.checkpw(password_bytes, database[username]):
        print("Login successful!")
    else:
        print("Login failed!")

# Emoji password
register("rakshit", "Pa$$word🔥✊🏻✊🏻☺️")
login("rakshit", "Pa$$word🔥✊🏻✊🏻☺️") # works

Registered successfully!
Login successful!


In [3]:
import hashlib
import requests

target_hash = "81a83544cf93c245178cbc1620030f1123f435af867c79d87135983c52ab39d9"

def dictionary_attack(url):
    print("Downloading wordlist...")
    
    response = requests.get(url)
    words = response.text.splitlines()
    
    print("Starting attack...")
    
    for word in words:
        word = word.strip()
        hashed = hashlib.sha256(word.encode()).hexdigest()
        
        if hashed == target_hash:
            print("Password found:", word)
            return word
    
    print("Password not found.")
    return None


url = "https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/10k-most-common.txt"

dictionary_attack(url)

Starting attack...
Password found: 2000


'2000'

In [1]:
#Exercise-4 -->
import hashlib
import itertools
SALT = "5UA@/Mw^%He]SBaU"
TARGET_HASH = "3281e6de7fa3c6fd6d6c8098347aeb06bd35b0f74b96f173c7b2d28135e14d45"
WORDS = [
    "Marie", "Curie", "MarieCurie",
    "Woof",
    "UKC", "Kent", "Canterbury",
    "Jean", "Neoskour", "JeanNeoskour",
    "Jvaist", "Fairecourir",
    "Eltrofor",
    "laplusbelle",
]
LEET_TABLE = str.maketrans("aeisoAEISO", "@31$0@31$0")
SUFFIXES = ["", "!", "@", "#", "1", "12", "123", "1234", "1!", "123!"]
SEPARATORS = ("", "-", "/", ".", "_")
def generate_date_variants(day, month, year):
    d, dd = str(day), f"{day:02d}"
    m, mm = str(month), f"{month:02d}"
    yy, yyyy = str(year)[2:], str(year)
    
    variants = {yyyy, yy, f"{dd}{mm}", f"{mm}{dd}", f"{d}{m}", f"{m}{d}"}
    
    for D, M, Y in itertools.product((d, dd), (m, mm), (yy, yyyy)):
        for sep in SEPARATORS:
            variants.update([
                f"{D}{sep}{M}{sep}{Y}",
                f"{M}{sep}{D}{sep}{Y}",
                f"{Y}{sep}{M}{sep}{D}"
            ])
    return variants
def generate_word_variants(word):
    bases = {word, word.lower(), word.upper(), word.capitalize(), word.title()}
    expanded = set()
    for b in bases:
        for sep in ("", "_", "-", ".", " "):
            s = b.replace(" ", sep)
            expanded.add(s)
            expanded.add(s.translate(LEET_TABLE))
    return expanded
def check_password(password):
    combinations = [password + SALT, SALT + password]
    for candidate in combinations:
        if hashlib.sha256(candidate.encode()).hexdigest() == TARGET_HASH:
            print(f"\n[+] MATCH FOUND: {password}")
            return True
    return False
def execute_attack():
    dob = generate_date_variants(2, 1, 1980)
    hbd = generate_date_variants(29, 12, 1981)
    all_dates = dob | hbd
    base_variants = set()
    for w in WORDS:
        base_variants.update(generate_word_variants(w))
    short_variants = [w for w in base_variants if len(w) <= 10]
    for w in base_variants:
        if check_password(w): return True
        for s in SUFFIXES[1:]:
            if check_password(w + s): return True
    for w, d in itertools.product(base_variants, all_dates):
        if check_password(w + d) or check_password(d + w): return True
        for sep in ("_", "-", ".", "@", "/"):
            if check_password(w + sep + d) or check_password(d + sep + w): return True
    for w, d, s in itertools.product(base_variants, all_dates, SUFFIXES[1:]):
        if check_password(w + d + s): return True
    for w1, w2 in itertools.permutations(short_variants, 2):
        combo = w1 + w2
        if 6 <= len(combo) <= 22:
            if check_password(combo): return True
            for s in SUFFIXES[1:]:
                if check_password(combo + s): return True
    for w1, w2 in itertools.permutations(short_variants, 2):
        for d in all_dates:
            if check_password(w1 + d + w2) or check_password(w2 + d + w1) or check_password(d + w1 + w2):
                return True
    return False
if __name__ == "__main__":
    execute_attack()


[+] MATCH FOUND: Woof122981Eltrofor


In [6]:
import hashlib
import itertools

target_hash = "b23a6a8439c0dde5515893e7c90c1e3233b8616e634470f20dc4928bcf3609bc"
points = "unknown"

def crack_pattern(max_length=9):
    for length in range(1, max_length + 1):
        print(f"Trying length {length}...")
        
        for pattern in itertools.permutations(points, length):
            pattern_str = "".join(pattern)
            hashed = hashlib.sha256(pattern_str.encode()).hexdigest()
            
            if hashed == target_hash:
                print("Pattern found:", pattern_str)
                return pattern_str

    print("Pattern not found.")
    return None

crack_pattern()

Trying length 1...
Trying length 2...
Trying length 3...
Trying length 4...
Trying length 5...
Trying length 6...
Trying length 7...
Pattern found: unknown


'unknown'